# The Bone Algorithm: Topology Optimization from Scratch
## SIMP in NumPy — no FEA library, no external optimizer

**What you will build:**
- Bilinear quad element stiffness matrix (KE) from first principles
- Global stiffness assembly with SIMP penalization
- Direct FEA solver + compliance sensitivity
- Cone-shaped sensitivity filter (removes checkerboard)
- Optimality Criteria (OC) bisection update
- Full cantilever benchmark + convergence plot
- Extension: MBB beam and custom boundary conditions

**Requirements:** numpy, scipy, matplotlib — all free on Colab CPU

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
print("All imports OK")

## 1. Element Stiffness Matrix

Each 2D bilinear quad element has 4 nodes × 2 DOFs = 8 DOFs.
The element stiffness matrix KE is derived analytically for a unit square (Sigmund 2001).

The key formula: KE = ∫ B^T C B dA
where B is the strain-displacement matrix and C is the elasticity tensor.

In [ ]:
def element_stiffness(nu=0.3):
    """8x8 element stiffness matrix for unit square bilinear quad.
    Derived from Sigmund (2001) — see paper for full derivation."""
    k = np.array([
        0.5 - nu/6,   0.125 + nu/8, -0.25 - nu/12, -0.125 + 3*nu/8,
       -0.25 + nu/12, -0.125 - nu/8,  nu/6,          0.125 - 3*nu/8
    ])
    KE = np.array([
        [k[0], k[1], k[2], k[3], k[4], k[5], k[6], k[7]],
        [k[1], k[0], k[7], k[6], k[5], k[4], k[3], k[2]],
        [k[2], k[7], k[0], k[5], k[6], k[3], k[4], k[1]],
        [k[3], k[6], k[5], k[0], k[7], k[2], k[1], k[4]],
        [k[4], k[5], k[6], k[7], k[0], k[1], k[2], k[3]],
        [k[5], k[4], k[3], k[2], k[1], k[0], k[7], k[6]],
        [k[6], k[3], k[4], k[1], k[2], k[7], k[0], k[5]],
        [k[7], k[2], k[1], k[4], k[3], k[6], k[5], k[0]],
    ]) / (1 - nu**2)
    return KE

KE = element_stiffness()
print(f"KE shape: {KE.shape}")
print(f"KE symmetric: {np.allclose(KE, KE.T)}")
print(f"KE positive definite (eigenvalues ≥ 0): {np.all(np.linalg.eigvalsh(KE) >= -1e-10)}")
print(f"KE trace (sum of diagonal): {np.trace(KE):.4f}")

## 2. DOF Mapping and Global Assembly

The domain is a grid of NELX × NELY quad elements.
Nodes are indexed column-major: node (col, row) → index col*(NELY+1) + row.
Each node has 2 DOFs (x, y displacement).

The global stiffness K is assembled by adding each element's KE
scaled by its SIMP-penalized Young's modulus.

In [ ]:
def build_dof_map(NELX, NELY):
    """Precompute 8 local DOF indices for every element."""
    nely1 = NELY + 1
    edofs = np.zeros((NELX * NELY, 8), dtype=int)
    for ely in range(NELY):
        for elx in range(NELX):
            e = ely * NELX + elx
            # 4 corner nodes: TL, TR, BR, BL
            nTL = nely1 * elx       + ely
            nTR = nely1 * (elx + 1) + ely
            nBR = nely1 * (elx + 1) + ely + 1
            nBL = nely1 * elx       + ely + 1
            edofs[e] = [2*nTL, 2*nTL+1, 2*nTR, 2*nTR+1,
                        2*nBR, 2*nBR+1, 2*nBL, 2*nBL+1]
    return edofs

def assemble_K(x, KE, edofs, NDOF, E0=1.0, EMIN=1e-3, penal=3.0):
    """Assemble global stiffness matrix using SIMP material model."""
    NEL = x.shape[0]
    rows, cols, vals = [], [], []
    for e in range(NEL):
        ke_scale = EMIN + x[e]**penal * (E0 - EMIN)
        d = edofs[e]
        for i in range(8):
            for j in range(8):
                rows.append(d[i]); cols.append(d[j])
                vals.append(ke_scale * KE[i, j])
    return sp.coo_matrix((vals, (rows, cols)), shape=(NDOF, NDOF)).tocsr()

# Quick sanity: 4×2 mesh
NELX_t, NELY_t = 4, 2
edofs_t = build_dof_map(NELX_t, NELY_t)
NDOF_t = 2 * (NELX_t+1) * (NELY_t+1)
x_t = np.ones(NELX_t * NELY_t)
K_t = assemble_K(x_t, KE, edofs_t, NDOF_t)
print(f"Test mesh {NELX_t}x{NELY_t}: NDOF={NDOF_t}, K shape={K_t.shape}, nnz={K_t.nnz}")

## 3. Boundary Conditions and FEA Solve

**Cantilever beam:** left edge (x=0) fully fixed, point load F downward at mid-right edge.

We apply BCs by removing fixed DOFs from the system before solving.
This is the "free DOF" approach — more numerically stable than penalty methods.

In [ ]:
def cantilever_bcs(NELX, NELY):
    """Returns (fixed_dofs, load_dof, load_val) for cantilever.
    Fixed: all nodes on left edge (x=0).
    Load: downward point load at mid-right node.
    """
    nely1 = NELY + 1
    fixed = []
    for row in range(nely1):
        node = row  # left column: node index = row
        fixed += [2*node, 2*node+1]
    fixed = np.array(sorted(set(fixed)), dtype=int)

    # Mid-right node
    load_node = NELX * nely1 + NELY // 2
    load_dof  = 2 * load_node + 1  # y-direction (downward)
    return fixed, load_dof, 1.0

def solve_fea(K, fixed_dofs, load_dof, load_val, NDOF):
    """Solve K_free * u_free = f_free and reconstruct full u."""
    all_dofs  = np.arange(NDOF)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    F = np.zeros(NDOF)
    F[load_dof] = load_val

    Kf = K[np.ix_(free_dofs, free_dofs)]
    ff = F[free_dofs]
    uf = spla.spsolve(Kf, ff)

    u = np.zeros(NDOF)
    u[free_dofs] = uf
    return u

# Quick test
NELX, NELY = 8, 4
NDOF = 2 * (NELX+1) * (NELY+1)
edofs = build_dof_map(NELX, NELY)
fixed, load_dof, load_val = cantilever_bcs(NELX, NELY)
x0 = np.full(NELX*NELY, 0.5)
K0 = assemble_K(x0, KE, edofs, NDOF)
u0 = solve_fea(K0, fixed, load_dof, load_val, NDOF)
compliance0 = load_val * u0[load_dof]
print(f"Test FEA: max displacement = {np.max(np.abs(u0)):.4f}")
print(f"Compliance = {compliance0:.4f}")

## 4. Sensitivity and Filter

**Sensitivity:** dC/dxe = -p * xe^(p-1) * (E0 - Emin) * ue^T KE ue

This is the derivative of compliance w.r.t. each element density.
Negative sign: adding material (increasing xе) decreases compliance (increases stiffness).

**Filter:** replace dc[e] with a cone-weighted average of its neighborhood.
This enforces a minimum length scale and eliminates checkerboard patterns.

In [ ]:
def compute_sensitivity(x, u, KE, edofs, E0=1.0, EMIN=1e-3, penal=3.0):
    """Compute compliance sensitivity dc/dxe for all elements."""
    NEL = x.shape[0]
    dc = np.zeros(NEL)
    compliance = 0.0
    for e in range(NEL):
        d  = edofs[e]
        ue = u[d]
        strain_energy = ue @ KE @ ue  # ue^T KE ue
        compliance   += x[e]**penal * strain_energy
        dc[e]         = -penal * max(x[e], 1e-10)**(penal-1) * (E0 - EMIN) * strain_energy
    return dc, compliance

def build_filter(NELX, NELY, rmin):
    """Precompute cone-shaped sensitivity filter weights."""
    NEL = NELX * NELY
    H  = [[] for _ in range(NEL)]
    Hs = np.zeros(NEL)
    for ely in range(NELY):
        for elx in range(NELX):
            e = ely * NELX + elx
            for ely2 in range(max(0, ely-int(np.ceil(rmin))-1),
                               min(NELY, ely+int(np.ceil(rmin))+1)):
                for elx2 in range(max(0, elx-int(np.ceil(rmin))-1),
                                   min(NELX, elx+int(np.ceil(rmin))+1)):
                    dist = np.sqrt((elx2-elx)**2 + (ely2-ely)**2)
                    w = max(0.0, rmin - dist)
                    if w > 0:
                        f = ely2 * NELX + elx2
                        H[e].append((f, w))
                        Hs[e] += w
    return H, Hs

def apply_filter(x, dc, H, Hs):
    """Apply sensitivity filter: smoothed dc."""
    dcf = np.zeros_like(dc)
    for e in range(len(dc)):
        num = sum(w * x[f] * abs(dc[f]) for f, w in H[e])
        dcf[e] = -num / (max(x[e], 1e-3) * Hs[e])
    return dcf

print("Filter and sensitivity functions defined")
# Precompute filter for the test mesh
H_t, Hs_t = build_filter(NELX, NELY, rmin=1.5)
dc_t, C_t = compute_sensitivity(x0, u0, KE, edofs)
dcf_t = apply_filter(x0, dc_t, H_t, Hs_t)
print(f"Raw dc range:      [{dc_t.min():.4f}, {dc_t.max():.4f}]")
print(f"Filtered dc range: [{dcf_t.min():.4f}, {dcf_t.max():.4f}]")

## 5. Optimality Criteria Update

The OC update finds the optimal density step that:
1. Moves material from low-sensitivity to high-sensitivity elements
2. Exactly satisfies the volume constraint (via bisection on lambda)
3. Limits the step size (move limit = 0.2)

The bisection is a 1D search — very fast, typically < 30 iterations.

In [ ]:
def oc_update(x, dcf, volfrac, move=0.2, l1=0.0, l2=1e9):
    """Optimality Criteria density update with volume bisection."""
    NEL = x.shape[0]
    xnew = np.zeros(NEL)
    while (l2 - l1) / (l1 + l2 + 1e-30) > 1e-4:
        lm = 0.5 * (l1 + l2)
        B  = np.sqrt(np.maximum(-dcf, 0) / lm)  # dV/dxe = 1 for all elements
        xc = x * B
        xnew = np.clip(xc, np.maximum(x - move, 1e-3),
                            np.minimum(x + move, 1.0))
        if xnew.mean() > volfrac:
            l1 = lm
        else:
            l2 = lm
    return xnew

# Quick OC test
xnew_t = oc_update(x0, dcf_t, volfrac=0.4)
print(f"Before OC: vol = {x0.mean():.4f}")
print(f"After OC:  vol = {xnew_t.mean():.4f}  (target: 0.40)")
print(f"Density range after update: [{xnew_t.min():.4f}, {xnew_t.max():.4f}]")

## 6. Full SIMP Loop

Now we put it all together. Each iteration:
1. Assemble K with current densities
2. Solve FEA: K u = F
3. Compute sensitivities
4. Filter sensitivities
5. OC density update
6. Check convergence (max density change < tol)

In [ ]:
def simp(NELX, NELY, volfrac=0.4, penal=3.0, rmin=1.5,
         max_iter=100, tol=0.01, verbose=True):
    """Full SIMP topology optimization.
    Returns density history and compliance history.
    """
    NEL  = NELX * NELY
    NDOF = 2 * (NELX + 1) * (NELY + 1)

    KE    = element_stiffness()
    edofs = build_dof_map(NELX, NELY)
    fixed, load_dof, load_val = cantilever_bcs(NELX, NELY)
    H, Hs = build_filter(NELX, NELY, rmin)

    x    = np.full(NEL, volfrac)
    xold = x.copy()
    compliance_hist = []
    density_hist    = [x.copy()]

    for it in range(1, max_iter + 1):
        K  = assemble_K(x, KE, edofs, NDOF, penal=penal)
        u  = solve_fea(K, fixed, load_dof, load_val, NDOF)
        dc, compliance = compute_sensitivity(x, u, KE, edofs, penal=penal)
        dcf  = apply_filter(x, dc, H, Hs)
        xnew = oc_update(x, dcf, volfrac)

        change = np.max(np.abs(xnew - x))
        compliance_hist.append(compliance)
        density_hist.append(xnew.copy())

        if verbose:
            print(f"It {it:3d}  C={compliance:.4f}  vol={xnew.mean():.4f}  "
                  f"change={change:.4f}")

        xold, x = x.copy(), xnew.copy()
        if change < tol:
            print(f"\nConverged at iteration {it}")
            break

    return np.array(density_hist).reshape(-1, NELY, NELX), np.array(compliance_hist)

# Small run first to verify
print("=== 8x4 cantilever (verification) ===")
hist_s, comp_s = simp(8, 4, volfrac=0.4, max_iter=30, verbose=True)
print(f"Final compliance: {comp_s[-1]:.4f}")

## 7. Visualize Results

Plot the final optimized density field and convergence history.

In [ ]:
def plot_result(hist, comp_hist, title="Topology Optimization"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Final density
    ax = axes[0]
    im = ax.imshow(hist[-1], cmap='Blues', vmin=0, vmax=1,
                   origin='upper', aspect='auto')
    ax.set_title(f'{title}\nFinal density field (iter {len(comp_hist)})')
    ax.set_xlabel('x elements'); ax.set_ylabel('y elements')
    plt.colorbar(im, ax=ax, label='density ρ')

    # Convergence
    ax = axes[1]
    ax.plot(comp_hist, color='#1A1AE5', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Compliance C(x)')
    ax.set_title('Convergence history')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

    plt.tight_layout()
    plt.savefig('topo_result.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved topo_result.png")

plot_result(hist_s, comp_s, title="8x4 Cantilever")

## 8. Scale Up to 60x30

Now run the classic benchmark at a larger resolution.
This takes 2-5 minutes on Colab CPU — a good time to read sections 5-7 of the article.

In [ ]:
print("=== 60x30 cantilever ===")
print("Running SIMP... (may take a few minutes on CPU)")
hist_L, comp_L = simp(60, 30, volfrac=0.4, penal=3.0, rmin=1.5, max_iter=100, verbose=False)
print(f"Done. {len(comp_L)} iterations, final C={comp_L[-1]:.4f}")
plot_result(hist_L, comp_L, title="60x30 Cantilever (full resolution)")

## 9. Animation: Watching the Structure Grow

Visualize how the optimizer evolves the density field from uniform to optimized.

In [ ]:
def animate_simp(hist, comp_hist, fps=8, title="SIMP evolution"):
    """Animate density evolution across iterations."""
    nframes = min(len(hist), 50)
    stride  = max(1, len(hist) // nframes)
    frames  = list(range(0, len(hist), stride))

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(hist[0], cmap='Blues', vmin=0, vmax=1,
                   origin='upper', aspect='auto')
    ax.set_title(title)
    txt = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                  color='red', fontsize=10, va='top')
    plt.colorbar(im, ax=ax, label='density')
    plt.tight_layout()

    def update(frame_idx):
        i = frames[frame_idx]
        im.set_data(hist[i])
        c = comp_hist[i] if i < len(comp_hist) else comp_hist[-1]
        txt.set_text(f'iter {i}  C={c:.3f}')
        return [im, txt]

    ani = animation.FuncAnimation(fig, update, frames=len(frames),
                                  interval=1000//fps, blit=True)
    plt.close()
    return HTML(ani.to_jshtml())

# Use the 60x30 result (or 8x4 if you skipped cell above)
try:
    anim = animate_simp(hist_L, comp_L, fps=10, title="60x30 Cantilever SIMP")
except NameError:
    anim = animate_simp(hist_s, comp_s, fps=5, title="8x4 Cantilever SIMP")
anim

## 10. Extension: MBB Beam

The MBB (Messerschmitt-Bolkow-Blohm) beam is the other classic benchmark.
Half-symmetry model: fix left corner vertically, fix right corner (roller), load at top-left.

In [ ]:
def mbb_bcs(NELX, NELY):
    """MBB beam boundary conditions (half model)."""
    fixed = []
    # Left edge: fix x-dof only (roller symmetry)
    for row in range(NELY + 1):
        node = row
        fixed.append(2 * node)        # x-direction only

    # Bottom-right corner: fix y-dof
    node_br = NELX * (NELY + 1) + NELY
    fixed.append(2 * node_br + 1)

    fixed = np.array(sorted(set(fixed)), dtype=int)

    # Load at top-left (downward)
    load_dof = 2 * 0 + 1   # node 0, y direction
    return fixed, load_dof, 1.0

def simp_custom_bcs(NELX, NELY, bc_fn, volfrac=0.5, penal=3.0,
                    rmin=1.5, max_iter=100, tol=0.01):
    """SIMP with custom boundary condition function."""
    NEL  = NELX * NELY
    NDOF = 2 * (NELX + 1) * (NELY + 1)
    KE    = element_stiffness()
    edofs = build_dof_map(NELX, NELY)
    fixed, load_dof, load_val = bc_fn(NELX, NELY)
    H, Hs = build_filter(NELX, NELY, rmin)

    x = np.full(NEL, volfrac)
    comp_hist = []
    density_hist = [x.copy()]

    for it in range(1, max_iter + 1):
        K = assemble_K(x, KE, edofs, NDOF, penal=penal)
        u = solve_fea(K, fixed, load_dof, load_val, NDOF)
        dc, compliance = compute_sensitivity(x, u, KE, edofs, penal=penal)
        dcf  = apply_filter(x, dc, H, Hs)
        xnew = oc_update(x, dcf, volfrac)
        comp_hist.append(compliance)
        density_hist.append(xnew.copy())
        change = np.max(np.abs(xnew - x))
        x = xnew.copy()
        if change < tol:
            print(f"Converged at iter {it}, C={compliance:.4f}")
            break

    return np.array(density_hist).reshape(-1, NELY, NELX), np.array(comp_hist)

print("=== 30x10 MBB beam ===")
hist_mbb, comp_mbb = simp_custom_bcs(30, 10, mbb_bcs, volfrac=0.5, max_iter=80)
plot_result(hist_mbb, comp_mbb, title="30x10 MBB Beam")

## 11. Effect of Penalization Exponent p

Higher p = sharper black/white boundary, but slower convergence.
Lower p = more grey (intermediate densities remain).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, p in zip(axes, [1.5, 3.0, 5.0]):
    print(f"Running p={p}...")
    hist_p, comp_p = simp(20, 10, volfrac=0.4, penal=p, rmin=1.5, max_iter=50, verbose=False)
    ax.imshow(hist_p[-1], cmap='Blues', vmin=0, vmax=1, origin='upper', aspect='auto')
    ax.set_title(f'p = {p}\nC = {comp_p[-1]:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
plt.suptitle('Effect of SIMP penalization exponent p', fontsize=13)
plt.tight_layout()
plt.savefig('simp_penalty_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Higher p = sharper 0/1 result; lower p = more grey (intermediate densities)")

## What's Next

You have built the complete SIMP algorithm from scratch. Extensions to explore:

**Manufacturing constraints**
- Overhang constraint (for 3D printing without support)
- Symmetry enforcement (left-right, top-bottom)
- Minimum hole size (Heaviside projection filter)

**Better solvers**
- Conjugate gradient + multigrid preconditioner (scales to millions of DOFs)
- GPU assembly with CuPy or Numba CUDA

**Advanced methods**
- Level-set topology optimization (smooth boundaries, CAD-ready)
- BESO (bi-directional evolutionary structural optimization)
- Neural topology optimization (NTopo, TopoDiff)
- 3D extension: replace bilinear quads with trilinear hexahedra

**Further reading**
- Sigmund 2001 — A 99 line topology optimization code in MATLAB
- Andreassen et al. 2011 — Efficient topology optimization using 88 lines of code
- Bendsoe & Sigmund 2004 — Topology Optimization: Theory, Methods and Applications

[Back to article](https://chenchihyuan.github.io/geo-instructor/posts/topology-optimization/)